In [ ]:
!pip install rank_bm25 faiss-cpu
!pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 73.2 MB/s eta 0:00:00


In [ ]:
!pip install -U "transformers>=4.41.0" "sentence-transformers>=2.7.0" \
              "peft>=0.10.0" "accelerate>=0.28.0" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.7 MB/s eta 0:00:00


In [ ]:
import os, random, torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.trainer import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from torch.utils.data import DataLoader
from transformers import EarlyStoppingCallback
import json
from pathlib import Path
from itertools import chain
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
from datasets import Dataset as HFDataset
import faiss
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, TrainingArguments, Trainer, set_seed
from google.colab import files
uploaded = files.upload()

Saving BioASQ-training13b.zip to BioASQ-training13b.zip


In [ ]:
zip_name = list(uploaded.keys())[0]
os.makedirs("/content/data/bioasq", exist_ok=True)
!unzip -q "$zip_name" -d /content/data/bioasq

!ls -l /content/data/bioasq | head

total 4
drwxrwxr-x 2 root root 4096 Oct  7  2024 BioASQ-training13b


In [ ]:
class CFG:
    seed            = 42
    zip_path        = Path("/content/BioASQ-training13b.zip")

    # working directories
    work_dir        = Path("/content/data")
    bioasq_dir      = work_dir / "bioasq"
    cache_dir       = work_dir / "cache"

    # misc paths
    hard_neg_cache  = cache_dir / "bm25_hard_negs.json"
    enc_name        = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

    # tokenisation / mining
    max_len         = 128
    num_negatives   = 5
    bm25_topk       = 50

    # training
    output_dir      = Path("outputs/retriever_fast")
    epochs          = 5
    lr              = 2e-5
    train_bsz       = 64
    eval_bsz        = 64
    warmup_ratio    = 0.1

In [ ]:
#reproducibility
set_seed(CFG.seed)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def _ensure_dirs():
    CFG.work_dir.mkdir(parents=True, exist_ok=True)
    CFG.cache_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_bioasq_pairs() -> tuple[list[tuple[str,str]], list[str]]:
    bioasq_dir = CFG.bioasq_dir / "BioASQ-training13b"
    pairs: list[tuple[str,str]] = []
    for jf in sorted(bioasq_dir.glob("*.json")):
        data = json.loads(jf.read_text())
        for q in data["questions"]:
            q_text = q["body"].strip()
            seen_ctx = set()
            for sn in q.get("snippets", []):
                ctx = sn["text"].strip()
                if ctx and ctx not in seen_ctx:
                    pairs.append((q_text, ctx))
                    seen_ctx.add(ctx)
    unique_ctx = sorted({c for _, c in pairs})
    print(f"Loaded {len(pairs):,} pairs | {len(unique_ctx):,} unique ctx")
    return pairs, unique_ctx

In [ ]:
# Hard‑negative mining w/ BM25
def mine_hard_negs(pairs: list[tuple[str,str]], unique_ctx: list[str]) -> dict[str, list[str]]:
    if CFG.hard_neg_cache.exists():
        return json.loads(CFG.hard_neg_cache.read_text())

    #build BM25 index
    bm25 = BM25Okapi([ctx.split() for ctx in unique_ctx])
    neg_cache: dict[str, list[str]] = {}

    for q, _ in tqdm(set(pairs), desc="BM25 mining"):
        cands = bm25.get_top_n(q.split(), unique_ctx, n=CFG.bm25_topk)
        negs  = [c for c in cands if c not in {ctx for qq, ctx in pairs if qq == q}]
        neg_cache[q] = negs[:CFG.num_negatives]

    CFG.hard_neg_cache.write_text(json.dumps(neg_cache))
    return neg_cache

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.enc_name, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def build_dataset(pairs: list[tuple[str,str]], neg_cache: dict[str, list[str]]):
    records = []
    for q, pos in pairs:
        negs = neg_cache.get(q, [])
        records.append({"query": q, "pos": pos, "neg": negs})

    ds = HFDataset.from_list(records)

    def tok_fn(ex):
        q_enc = tokenizer(ex["query"], truncation=True, max_length=CFG.max_len)
        p_enc = tokenizer(ex["pos"],   truncation=True, max_length=CFG.max_len)
        n_encs = [tokenizer(n, truncation=True, max_length=CFG.max_len) for n in ex["neg"]]
        return {"q": q_enc, "p": p_enc, "n": n_encs}

    ds = ds.map(tok_fn, batched=False, num_proc=4, desc="tokenise")
    ds = ds.shuffle(seed=CFG.seed)
    val_size = int(0.1 * len(ds))
    val_ds = ds.select(range(val_size))
    train_ds = ds.select(range(val_size, len(ds)))
    return train_ds, val_ds

In [ ]:
def collate(features):
    batch_q = tokenizer.pad([f["q"] for f in features], return_tensors="pt")
    batch_p = tokenizer.pad([f["p"] for f in features], return_tensors="pt")

    # flatten negatives
    neg_flat = list(chain.from_iterable(f["n"] for f in features))
    offsets  = [0]
    for f in features:
        offsets.append(offsets[-1] + len(f["n"]))

    if neg_flat:
        batch_n = tokenizer.pad(neg_flat, return_tensors="pt")
    else:
        batch_n = None

    return dict(q_tok=batch_q, pos_tok=batch_p, neg_tok=batch_n, offsets=offsets)

In [ ]:
class BiEncoder(nn.Module):
    def __init__(self, encoder: SentenceTransformer, temperature: float = 0.05):
        super().__init__()
        self.enc  = encoder
        self.temp = temperature

    def _encode(self, tok):
        feats = {"input_ids": tok["input_ids"],
                 "attention_mask": tok["attention_mask"]}
        if "token_type_ids" in tok and tok["token_type_ids"] is not None:
            feats["token_type_ids"] = tok["token_type_ids"]
        emb = self.enc(feats)["sentence_embedding"]
        return F.normalize(emb, p=2, dim=-1)

    def forward(self, q_tok, pos_tok, neg_tok=None, offsets=None):
        q   = self._encode(q_tok)
        pos = self._encode(pos_tok)
        docs = [pos]
        if neg_tok is not None:
            docs.append(self._encode(neg_tok))
        docs  = torch.cat(docs, 0)
        logits = q @ docs.T / self.temp
        labels = torch.arange(q.size(0), device=q.device)
        return {"loss": F.cross_entropy(logits, labels)}

In [ ]:
def build_trainer(train_ds, val_ds, unique_ctx):
    sbert = SentenceTransformer(CFG.enc_name)
    model = BiEncoder(sbert, temperature=0.05)

    base_model = model.enc._first_module().auto_model
    if hasattr(base_model, "gradient_checkpointing_enable"):
        base_model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir               = str(CFG.output_dir),
        num_train_epochs         = CFG.epochs,
        per_device_train_batch_size = CFG.train_bsz,
        per_device_eval_batch_size  = CFG.eval_bsz,
        learning_rate            = CFG.lr,
        warmup_ratio             = CFG.warmup_ratio,
        weight_decay             = 0.01,
        bf16                     = torch.cuda.is_available(),
        lr_scheduler_type        = "cosine",
        max_grad_norm           = 1.0,
        eval_strategy            = "steps",
        eval_steps               = 1000,
        save_strategy            = "steps",
        save_steps               = 1000,
        save_total_limit         = 2,
        remove_unused_columns    = False,
        logging_steps            = 50,
        report_to                = ["wandb"],
        run_name                 = "bioasq_biencoder",
        seed                     = CFG.seed,
    )

    optim = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=0.01, fused=True)

    # metrics
    val_cache = {}

    @torch.no_grad()
    def compute_val_metrics(_):
        if not val_cache:
            ctx_emb = model.embed(unique_ctx, bs=256).cpu()
            val_cache["ctx_emb"] = ctx_emb
            index = faiss.IndexFlatIP(ctx_emb.shape[1])
            index.add(ctx_emb.numpy())
            val_cache["index"] = index
        if "q_emb" not in val_cache:
            val_cache["q_emb"] = model.embed(list(val_ds["query"]), bs=256).cpu()

        D, I = val_cache["index"].search(val_cache["q_emb"].numpy(), 10)
        hits = {1:0,5:0,10:0}
        mrr  = 0.0
        ctx2id = {c:i for i,c in enumerate(unique_ctx)}
        for qi, pos in enumerate(val_ds["pos"]):
            pos_id = ctx2id[pos]
            retrieved = I[qi]
            for k in hits:
                if pos_id in retrieved[:k]:
                    hits[k] += 1
            for rank, r in enumerate(retrieved, 1):
                if r == pos_id:
                    mrr += 1.0 / rank
                    break
        n = len(val_ds)
        return {f"r_at_{k}": hits[k]/n for k in hits} | {"mrr": mrr/n}

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        data_collator   = collate,
        compute_metrics = compute_val_metrics,
        optimizers      = (optim, None),
    )
    return trainer, model

In [ ]:
def save_index(model: SentenceTransformer, unique_ctx: list[str]):
    emb = model.encode(unique_ctx, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)

    CFG.output_dir.mkdir(parents=True, exist_ok=True)
    model.save(str(CFG.output_dir / "sentence_transformer"))
    faiss.write_index(index, str(CFG.output_dir / "bioasq_flatip_cosine.idx"))

In [ ]:
def main():
    _ensure_dirs()

    pairs, unique_ctx = load_bioasq_pairs()
    neg_cache         = mine_hard_negs(pairs, unique_ctx)
    train_ds, val_ds  = build_dataset(pairs, neg_cache)

    trainer, biencoder = build_trainer(train_ds, val_ds, unique_ctx)
    trainer.train()

    save_index(biencoder.enc, unique_ctx)

    return val_ds, unique_ctx


if __name__ == "__main__":
    val_ds, unique_ctx = main()

Loaded 65,758 pairs | 64,736 unique ctx


BM25 mining:   0%|          | 0/65758 [00:00<?, ?it/s]

tokenise (num_proc=4):   0%|          | 0/65758 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1395: FutureWarning: promote has been superseded by promote_options='default'.
  block_group = [InMemoryTable(cls._concat_blocks(list(block_group), axis=axis))]
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vkuruppu23 (vkuruppu23-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
1000,0.357800,No log
2000,0.203100,No log
3000,0.148800,No log
4000,0.132200,No log


# Eval

In [ ]:
def score_retriever(model: SentenceTransformer,
                    queries: list[str],
                    pos_ctxs: list[str],
                    corpus: list[str],
                    k: int = 10,
                    bsz: int = 256) -> dict[str, float]:
    """
    Compute Recall@{1,5,10} and MRR for (queries, pos_ctxs) over *corpus*.
    """
    # embed corpus
    ctx_emb = model.encode(corpus,
                           batch_size=bsz,
                           convert_to_numpy=True,
                           normalize_embeddings=True).astype(np.float32)
    index = faiss.IndexFlatIP(ctx_emb.shape[1])
    index.add(ctx_emb)

    # embed queries
    q_emb = model.encode(queries,
                         batch_size=bsz,
                         convert_to_numpy=True,
                         normalize_embeddings=True).astype(np.float32)

    # search
    D, I = index.search(q_emb, k)   # (num_q, k)

    # metrics
    id_lookup = {c: i for i, c in enumerate(corpus)}
    hits1 = hits5 = hits10 = mrr = ndcg =0.0

    for qi, pos in enumerate(pos_ctxs):
        pos_id   = id_lookup[pos]
        retrieved = I[qi]

        #Recall @ 1,5 ,10
        if pos_id in retrieved[:1]:
            hits1 += 1
        if pos_id in retrieved[:5]:
            hits5 += 1
        if pos_id in retrieved[:10]:
            hits10 += 1

        # reciprocal rank
        # for rank, idx in enumerate(retrieved, 1):
        #     if idx == pos_id:
        #         mrr += 1.0 / rank
        #         break
        for rank, idx in enumerate(retrieved, start=1):
            if idx == pos_id:
                mrr  += 1.0 / rank
                # DCG for binary relevance = 1/log2(rank+1)
                ndcg += 1.0 / math.log2(rank + 1)
                break


    n = len(queries)
    return {
        "R@1":  hits1  / n,
        "R@5":  hits5  / n,
        "R@10": hits10 / n,
        "MRR":  mrr    / n,
        "nDCG@{}".format(k): ndcg / n,
    }

In [ ]:
# baseline BioBERT (no fine‑tuning)
base_ckpt  = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
baseline_st = SentenceTransformer(base_ckpt)

# fine‑tuned bi‑encoder checkpoint
ft_path    = "outputs/retriever_fast/sentence_transformer"
finetuned_st = SentenceTransformer(ft_path)

In [ ]:
queries   = list(val_ds["query"])
positives = list(val_ds["pos"])
corpus    = unique_ctx

baseline_scores  = score_retriever(baseline_st,  queries, positives, corpus)
finetuned_scores = score_retriever(finetuned_st, queries, positives, corpus)

print("Baseline :", baseline_scores)
print("Finetuned:", finetuned_scores)

Baseline : {'R@1': 0.057338403041825095, 'R@5': 0.21323193916349809, 'R@10': 0.339467680608365, 'MRR': 0.12657619651155788}
Finetuned: {'R@1': 0.05885931558935361, 'R@5': 0.2485171102661597, 'R@10': 0.41642585551330796, 'MRR': 0.14440129156859222}


In [ ]:
!zip -r retriever_finetuned.zip outputs/retriever_fast/sentence_transformer

from google.colab import files
files.download('retriever_finetuned.zip')

  adding: outputs/retriever_fast/sentence_transformer/ (stored 0%)
  adding: outputs/retriever_fast/sentence_transformer/tokenizer_config.json (deflated 74%)
  adding: outputs/retriever_fast/sentence_transformer/tokenizer.json (deflated 70%)
  adding: outputs/retriever_fast/sentence_transformer/model.safetensors (deflated 7%)
  adding: outputs/retriever_fast/sentence_transformer/config_sentence_transformers.json (deflated 40%)
  adding: outputs/retriever_fast/sentence_transformer/README.md (deflated 58%)
  adding: outputs/retriever_fast/sentence_transformer/config.json (deflated 48%)
  adding: outputs/retriever_fast/sentence_transformer/vocab.txt (deflated 49%)
  adding: outputs/retriever_fast/sentence_transformer/1_Pooling/ (stored 0%)
  adding: outputs/retriever_fast/sentence_transformer/1_Pooling/config.json (deflated 59%)
  adding: outputs/retriever_fast/sentence_transformer/special_tokens_map.json (deflated 80%)
  adding: outputs/retriever_fast/sentence_transformer/sentence_bert_c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>